# Interview drills (60 minutes)

Each drill maps to the README follow-up or a common live question:

| Drill | What I practice |
|-------|------------------|
| 1 | Run the frozen model on a **new event stream** |
| 2 | **Investigate a failed invariant** (late correction) |
| 3 | **Change a requirement** (horizon + split policy) |
| 4 | **Defend evaluation** (split, baseline, carrier slices) |
| 5 | **Small code change** without breaking replay determinism |
| 6 | **Concurrency + customer notes** (thread smoke test, rejections table) |
| 7 | **Pre-interview checklist** (pytest, artifacts, docs) |

```bash
jupyter notebook optional/notebooks/interview_drills.ipynb
pytest -v
```


In [ ]:
import importlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

OPTIONAL_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "optional"
ROOT = OPTIONAL_DIR.parent
sys.path[:0] = [str(ROOT / "src"), str(OPTIONAL_DIR)]

DATA_DIR = ROOT / "data"
ARTIFACT_DIR = ROOT / "artifact"
INTERVIEW_DIR = OPTIONAL_DIR / "data" / "interview_stream"

from dispatch_risk.contracts import TelemetryEvent
from dispatch_risk.solution import (
    RiskEngine,
    _build_features,
    _events_known_at,
    build_training_rows,
    train,
)

import demo_helpers

importlib.reload(demo_helpers)
from demo_helpers import (
    compare_evaluation_splits,
    compare_reason_patch,
    compare_time_based_split,
    configure_notebook_style,
    customer_requirement_table,
    demo_concurrency_smoke,
    demo_late_correction,
    events_known_at_wrong_policy,
    generate_fresh_stream,
    holdout_carrier_slices,
    holdout_evaluation,
    ingest_delay_summary,
    label_horizon_summary,
    load_decision_times,
    load_events,
    load_labels,
    plot_correction_before_after,
    plot_holdout_panels,
    replay_wire_outputs,
    run_determinism_pytests,
    run_interview_checklist,
    verify_deterministic_replay,
)

configure_notebook_style()
print(f"Project root: {ROOT}")


In [ ]:
# Train once if the artifact is missing
if not (DATA_DIR / "events.jsonl").exists():
    subprocess.run([sys.executable, "tools/generate_dataset.py"], check=True, cwd=ROOT)

if not (ARTIFACT_DIR / "model.joblib").exists():
    rows = build_training_rows(
        load_events(DATA_DIR),
        load_labels(DATA_DIR),
        load_decision_times(DATA_DIR),
    )
    metrics = train(rows, ARTIFACT_DIR)
    print("Trained model because artifact/ was empty")
else:
    metrics = json.loads((ARTIFACT_DIR / "metrics.json").read_text())
    print(f"Loaded artifact: {metrics['selected_model']} PR-AUC {metrics['pr_auc']:.3f}")


## Architecture cheat sheet

**Training:** events → `received_at` cutoff → features → **shipment split** → logreg vs HGB → artifact

**Serving:** ingest (idempotent) → revision pick → LRU eviction → score → deterministic wire JSON

**Five answers to memorize**
1. **Knowledge cutoff:** `received_at`, not `device_time`
2. **Evaluation:** hold out later shipments (by first decision time); report PR-AUC + Brier + baseline + slices
3. **Late corrections:** highest revision with `received_at <= as_of`; past scores unchanged
4. **Concurrency:** one `RLock` on ingest, score, snapshot, restore, reload
5. **Customer notes:** see Drill 6 table (10 items in `DECISIONS.md`)

---
## Drill 1: New event stream

The interviewer generates data I have never seen. I keep the trained artifact, ingest in delivery order, and score at the provided decision times.

Change `INTERVIEW_SEED` to simulate a different stream.

In [ ]:
INTERVIEW_SEED = 9999
INTERVIEW_SHIPMENTS = 120

manifest = generate_fresh_stream(
    INTERVIEW_DIR,
    seed=INTERVIEW_SEED,
    shipments=INTERVIEW_SHIPMENTS,
)
print(
    f"Fresh stream seed {manifest['seed']}: "
    f"{manifest['events']} events, {manifest['labels']} labels, "
    f"{manifest['decision_times']} decision times"
)

stream_events = load_events(INTERVIEW_DIR)
stream_labels = load_labels(INTERVIEW_DIR)
stream_decisions = load_decision_times(INTERVIEW_DIR)

engine = RiskEngine(ARTIFACT_DIR, max_shipments=32)
sample_decisions = stream_decisions[:12]
sample_wires = replay_wire_outputs(engine, stream_events, sample_decisions)

preview = []
for (shipment_id, as_of), wire in zip(sample_decisions, sample_wires, strict=True):
    pred = engine.score(shipment_id, as_of)
    preview.append(
        {
            "shipment_id": shipment_id,
            "as_of": as_of.strftime("%Y-%m-%d %H:%M UTC"),
            "probability": round(pred.probability, 3),
            "reasons": ", ".join(pred.reasons),
            "model_version": pred.model_version,
        }
    )
display(pd.DataFrame(preview))

replay_check = verify_deterministic_replay(
    ARTIFACT_DIR,
    stream_events,
    stream_decisions[:30],
)
print(json.dumps(replay_check, indent=2))


**Talking points:** I never retrain on the interview stream. I load the artifact, respect delivery order, and treat duplicate `(event_id, revision)` as no-ops. Identical replays must match byte-for-byte.

---
## Drill 2: Investigate a failed invariant

Invariant: a late correction must **not** change a score already issued at an earlier `as_of`.

If this fails, I trace `_events_known_at` and check whether `received_at <= as_of` is applied.

In [ ]:
from demo_helpers import WALKTHROUGH_DIR, write_walkthrough_dataset

if not (WALKTHROUGH_DIR / "events.jsonl").exists():
    write_walkthrough_dataset(WALKTHROUGH_DIR, seed=4242)

before, after, as_of_past = demo_late_correction(ARTIFACT_DIR, WALKTHROUGH_DIR)
passed = before.to_wire() == after.to_wire()

print(f"Score as of {as_of_past.strftime('%H:%M UTC')}")
print(f"Before correction: {before.probability:.1%}  reasons={before.reasons}")
print(f"After correction:  {after.probability:.1%}  reasons={after.reasons}")
print(f"Invariant held: {passed}")

fig = plot_correction_before_after(before.probability, after.probability, as_of_label=as_of_past.strftime("%H:%M UTC"))
plt.show()


In [ ]:
# Broken policy from customer note #2: always take the newest revision.
original = TelemetryEvent(
    event_id="evt-debug",
    revision=1,
    shipment_id="s-debug",
    device_time=datetime(2026, 1, 1, 8, tzinfo=timezone.utc),
    received_at=datetime(2026, 1, 1, 8, 5, tzinfo=timezone.utc),
    kind="temperature_c",
    value=9.0,
    source="sensor-north",
    payload={},
)
correction = TelemetryEvent(
    event_id="evt-debug",
    revision=2,
    shipment_id="s-debug",
    device_time=datetime(2026, 1, 1, 8, tzinfo=timezone.utc),
    received_at=datetime(2026, 1, 1, 18, tzinfo=timezone.utc),
    kind="temperature_c",
    value=3.0,
    source="sensor-north",
    payload={"correction": True},
)
as_of = datetime(2026, 1, 1, 10, tzinfo=timezone.utc)
events = [original, correction]

correct_known = _events_known_at(events, as_of)
wrong_known = events_known_at_wrong_policy(events, as_of)

correct_features = _build_features(correct_known)
wrong_features = _build_features(wrong_known)

display(
    pd.DataFrame(
        [
            {"policy": "correct (received_at cutoff)", **{k: correct_features[k] for k in correct_features}},
            {"policy": "broken (newest revision always)", **{k: wrong_features[k] for k in wrong_features}},
        ]
    )
)

engine = RiskEngine(ARTIFACT_DIR, max_shipments=4)
engine.ingest(original)
before_wire = engine.score("s-debug", as_of).to_wire()
engine.ingest(correction)
after_wire = engine.score("s-debug", as_of).to_wire()
print(f"Production engine keeps past score: {before_wire == after_wire}")
print(f"Wrong policy would change latest_temp_c: {correct_features['latest_temp_c']} -> {wrong_features['latest_temp_c']}")


**Talking points:** The bug shows up as a feature change at a fixed `as_of`. My fix is knowledge-time cutoff on `received_at`, documented in `DECISIONS.md`.

---
## Drill 3: Modify one requirement

Example ask: "Operations wants a 4-hour window instead of 6 hours."

I locate `HORIZON` in `src/dispatch_risk/solution.py`, retrain, and show how labels shift before deploying.

In [ ]:
training_rows = build_training_rows(
    load_events(DATA_DIR),
    load_labels(DATA_DIR),
    load_decision_times(DATA_DIR),
)

horizon_change = label_horizon_summary(
    load_labels(DATA_DIR),
    load_decision_times(DATA_DIR),
    old_hours=6,
    new_hours=4,
)
print(
    f"Positives at 6h: {horizon_change['positives_6h']}  "
    f"Positives at 4h: {horizon_change['positives_4h']}  "
    f"Rows that flip: {len(horizon_change['flipped_rows'])}"
)

if horizon_change["flipped_rows"]:
    display(pd.DataFrame(horizon_change["flipped_rows"]).head(10))
else:
    print("No label flips on this seed (try another seed if you want examples).")

print("\nSplit policy comparison (logreg on each split):")
split_policy = compare_time_based_split(training_rows)
display(pd.DataFrame([split_policy]))
print(
    f"Time-based test rows from shipments already in train: "
    f"{split_policy['test_rows_leaking_shipments']}/{split_policy['test_row_count']}"
)


**Code change checklist**

1. Update `HORIZON` and the docstrings that mention 6 hours.
2. Retrain with `train(build_training_rows(...), ARTIFACT_DIR)`.
3. Re-run `pytest` (label window tests live in `tests/test_solution.py::TestTrainingLabels`).
4. Confirm replay determinism still passes.

---
## Drill 4: Defend statistical validity

I split by **shipment**, not by row. Rows from the same route share telemetry and would leak if scattered across train and test.

I report **PR-AUC** because incidents are rare, plus **Brier score** for calibration and a constant baseline.

In [ ]:
print(json.dumps(
    {
        "split": metrics["evaluation_split"],
        "train_rows": metrics["train_rows"],
        "test_rows": metrics["test_rows"],
        "selected_model": metrics["selected_model"],
        "pr_auc": round(metrics["pr_auc"], 3),
        "brier_score": round(metrics["brier_score"], 4),
        "baseline_pr_auc": round(metrics["baseline_pr_auc"], 3),
    },
    indent=2,
))

split_compare = compare_evaluation_splits(training_rows)
display(pd.DataFrame([split_compare]))

holdout = holdout_evaluation(training_rows, ARTIFACT_DIR)
fig = plot_holdout_panels(holdout)
plt.show()


In [ ]:
carrier_slices = holdout_carrier_slices(training_rows, load_events(DATA_DIR), ARTIFACT_DIR)
display(pd.DataFrame(carrier_slices))

print(
    "If a slice looks weak, I check sample size and positive rate before blaming the model. "
    "I would not launch on PR-AUC alone without calibration and a baseline beat."
)


**Talking points:** Random row splits inflate metrics because the same shipment appears in both sets. Shipment holdout approximates scoring a route we have never seen. Baseline PR-AUC shows lift over predicting the training prevalence everywhere.

---
## Drill 5: Small code change, keep replay deterministic

**Scenario:** Ops wants a new explainability tag when a shipment has many temperature readings.

**Change:** add `dense_readings` to `_prediction_reasons` when `temp_reading_count >= 8`.

**Rules for a safe change**
- OK: extra reason codes derived from existing features (probability and `feature_digest` stay the same)
- OK: wire bytes change, but two replays with the same code must still match
- Not OK: randomness, wall-clock time, or changing feature math without retraining

Step 1 runs the patch in a **sandbox** (no file edit). Step 2 applies it to `solution.py` for real.

In [ ]:
from demo_helpers import (
    INTERVIEW_DENSE_READINGS_PATCH,
    WALKTHROUGH_DIR,
    apply_dense_readings_reason_patch,
    compare_reason_patch,
    decisions_for_shipment,
    write_walkthrough_dataset,
)
from IPython.display import Markdown

if not (WALKTHROUGH_DIR / "events.jsonl").exists():
    write_walkthrough_dataset(WALKTHROUGH_DIR, seed=4242)

PATCH_SHIPMENT = "s-demo-warming"
patch_events = load_events(WALKTHROUGH_DIR)
patch_decisions = decisions_for_shipment(load_decision_times(WALKTHROUGH_DIR), PATCH_SHIPMENT)

display(Markdown(f"```python\n{INTERVIEW_DENSE_READINGS_PATCH}\n```"))

patch_report = compare_reason_patch(ARTIFACT_DIR, patch_events, patch_decisions)
changed = patch_report["comparison"]
changed = changed[changed["reasons_changed"]].copy()

print(
    f"Shipment {PATCH_SHIPMENT}: {patch_report['rows_with_new_reason']} decision times gain dense_readings"
)
print(f"Probability unchanged: {patch_report['probability_unchanged']}")
print(f"Feature digest unchanged: {patch_report['feature_digest_unchanged']}")

if changed.empty:
    print("No reason changes on this shipment (pick another stream in the cell above).")
else:
    display(
        changed[
            [
                "as_of",
                "probability_before",
                "probability_after",
                "feature_digest_before",
                "reasons_before",
                "reasons_after",
            ]
        ]
    )

with apply_dense_readings_reason_patch():
    replay_after_patch = verify_deterministic_replay(
        ARTIFACT_DIR,
        patch_events,
        patch_decisions,
    )
print("Replay with sandbox patch:")
print(json.dumps(replay_after_patch, indent=2))


In [ ]:
# Step 2: after you paste the patch into solution.py, reload and verify.
from demo_helpers import run_determinism_pytests

solution_path = ROOT / "src" / "dispatch_risk" / "solution.py"
source = solution_path.read_text()

if "dense_readings" in source:
    import dispatch_risk.solution as solution_module

    importlib.reload(solution_module)
    print("Reloaded solution.py (dense_readings patch detected on disk).")

    live_report = compare_reason_patch(ARTIFACT_DIR, patch_events, patch_decisions)
    print(
        f"Live code: {live_report['rows_with_new_reason']} rows with dense_readings, "
        f"replay ok={live_report['replay_after_patch']['prediction_runs_match']}"
    )
else:
    print("Patch not on disk yet. Add dense_readings to _prediction_reasons, then re-run this cell.")

result = run_determinism_pytests(ROOT)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print("Determinism tests passed.")


**Interview checklist**

1. Make the smallest change that satisfies the ask (here: two lines in `_prediction_reasons`).
2. Confirm model outputs you care about are unchanged (`probability`, `feature_digest`).
3. Run identical replay twice; wire bytes must match across runs with the same code.
4. Run `pytest` determinism tests before you say you are done.

**If determinism breaks**, look for unordered iteration, timestamps in snapshots, or non-deterministic feature code.

---
## Drill 6: Concurrency and customer notes

**Concurrency:** one reentrant lock wraps ingest, score, snapshot, restore, and reload. The smoke test below mirrors `test_concurrent_ingest_and_score`.

**Delayed telemetry:** ingest delay summary shows why `received_at` matters.

**Customer notes:** be ready to defend all 10 rejections from the take-home brief.

In [ ]:
concurrency = demo_concurrency_smoke(ARTIFACT_DIR)
print(json.dumps({k: v for k, v in concurrency.items() if k != "errors"}, indent=2))
if concurrency["errors"]:
    print("Errors:", concurrency["errors"])

delays = ingest_delay_summary(load_events(DATA_DIR))
print("\nIngest delay (received_at - device_time), minutes:")
print(json.dumps(delays, indent=2))

display(customer_requirement_table())

from demo_helpers import INTERVIEW_FEATURE_PATCH
from IPython.display import Markdown

display(Markdown(f"**Feature-add pattern (practice, do not apply unless asked):**\n```python\n{INTERVIEW_FEATURE_PATCH}\n```"))

**Scenario responses**

- **Performance drops on new data:** check carrier/route drift, delay patterns, retrain pipeline; shipment split estimates generalization
- **Max shipments too low:** LRU evicts oldest; evicted shipments lose in-memory history; could prioritize by carrier if product requires it
- **Events arrive late:** already handled via `received_at`; monitor p95 delay; may widen decision windows operationally

---
## Drill 7: Pre-interview checklist

Run this the morning of the interview.

In [ ]:
checklist = run_interview_checklist(ROOT)
for item, ok in checklist.items():
    if item == "pytest_summary":
        continue
    print(f"{'[x]' if ok else '[ ]'} {item}")

print(f"\npytest: {checklist['pytest_summary']}")
if not checklist["pytest_passed"]:
    print("Fix failing tests before the interview.")
else:
    print("Ready. Review DECISIONS.md and re-run the drills above.")